 this is trial code


In [1]:
!pip install mne==1.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 34.8 MB/s eta 0:00:00


In [2]:
import mne
def convertDF2MNE(sub):
    info = mne.create_info(list(sub.columns), ch_types=['eeg'] * len(sub.columns), sfreq=128)
    info.set_montage('standard_1020')
    data = mne.io.RawArray(sub.T, info)
    data.set_eeg_reference() ## sir

    # Alternative filtering approach
    # Method A: Use a different filtering method
    data.filter(l_freq=1, h_freq=30, method='iir')  # Use IIR filter instead of FIR

    # OR Method B: If above still fails, use raw array filtering
    # from scipy import signal
    # data._data = signal.filtfilt(b, a, data._data, axis=1)

    epochs = mne.make_fixed_length_epochs(data, duration=5, overlap=1)
    epochs = epochs.drop_bad()
    return epochs

/usr/local/lib/python3.12/dist-packages/mne/datasets/eegbci/eegbci.py:8: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [3]:
from google.colab import files
uploaded = files.upload()

Saving EEGs_Nigeria (1).zip to EEGs_Nigeria (1).zip


In [4]:
from zipfile import ZipFile
data = ZipFile('/content/EEGs_Nigeria (1).zip')
data.extractall()

In [5]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [6]:
meta_df = pd.read_csv('https://zenodo.org/record/1252141/files/metadata_nigeria.csv')
meta_df.head()

,subject.id,recordedPeriod,startTime,session.id,first_condition,remarks,Group,csv.file
0,6,270,26/9/2016 13:13,1,open,NaN,control,signal-6-1.csv.gz
1,9,271,26/9/2016 13:30,1,closed,NaN,control,signal-9-1.csv.gz
2,10,272,26/9/2016 13:36,1,open,eyes closed at 2:40,control,signal-10-1.csv.gz
3,11,274,26/9/2016 13:42,2,closed,no.11.1 failed >> 11.2 is the right one,control,signal-11-2.csv.gz
4,11,1,26/9/2016 13:42,1,closed,no.11.1 failed >> 11.2 is the right one,control,signal-11-1.csv.gz


In [7]:
#now i need to seprate Epilepsy vs Control subjects
EP_sub = meta_df['subject.id'][meta_df['Group']=='epilepsy']
CT_sub = meta_df['subject.id'][meta_df['Group']=='control']

In [8]:
def appender(items,limit):
  iter = 0
  item = []
  for i in items:
    if (iter==limit):
      break
    try:
      reader = pd.read_csv('EEGs_Nigeria/signal-{}-1.csv.gz'.format(i), compression='gzip')
      item.append(reader)
      iter=iter+1
    except:
      try:
        reader = pd.read_csv('EEGs_Nigeria/signal-{}-2.csv.gz'.format(i), compression='gzip')
        item.append(reader)
        iter=iter+1
      except:
        pass
  return item
Epilepsy = appender(EP_sub,51)
Control = appender(CT_sub,46)

In [9]:
len(Control)+len(Epilepsy),Epilepsy[0].head()

(97,
    Unnamed: 0   AF3         AF4    F3          F4          F7    F8  \
 0           1  4036  330.221053  4134 -213.473684  155.242105  3712   
 1           2  4053  341.505263  4138 -211.936842  160.368421  3715   
 2           3  4053 -344.757895  4137 -212.957895  156.778947  3713   
 3           4  4044  343.557895  4136 -212.452632  147.042105  3713   
 4           5  4049 -344.242105  4131 -213.989474  141.400000  3712   
 
           FC5   FC6    O1  ...  CQ_F3  CQ_P7  CQ_P8  CQ_F4  CQ_AF3  CQ_FC5  \
 0  322.421053  3935  4169  ...      4      4      4      4       4       4   
 1  321.905263  3944  4173  ...      4      4      4      4       4       4   
 2  322.421053  3939  4174  ...      4      4      4      4       4       4   
 3  323.452632  3930  4170  ...      4      4      4      4       4       4   
 4  319.347368  3927  4170  ...      4      4      4      4       4       4   
 
    CQ_O1  CQ_T8  CQ_F8  CQ_DRL  
 0      4      4      4       4  
 1      4      4 

In [10]:
Epilepsy2 = [i.iloc[:,1:14] for i in  Epilepsy] ### maybe 15
Control2 = [i.iloc[:,1:14] for i in  Control]
Epilepsy[0].head()

,Unnamed: 0,AF3,AF4,F3,F4,F7,F8,FC5,FC6,O1,...,CQ_F3,CQ_P7,CQ_P8,CQ_F4,CQ_AF3,CQ_FC5,CQ_O1,CQ_T8,CQ_F8,CQ_DRL
0,1,4036,330.221053,4134,-213.473684,155.242105,3712,322.421053,3935,4169,...,4,4,4,4,4,4,4,4,4,4
1,2,4053,341.505263,4138,-211.936842,160.368421,3715,321.905263,3944,4173,...,4,4,4,4,4,4,4,4,4,4
2,3,4053,-344.757895,4137,-212.957895,156.778947,3713,322.421053,3939,4174,...,4,4,4,4,4,4,4,4,4,4
3,4,4044,343.557895,4136,-212.452632,147.042105,3713,323.452632,3930,4170,...,4,4,4,4,4,4,4,4,4,4
4,5,4049,-344.242105,4131,-213.989474,141.400000,3712,319.347368,3927,4170,...,4,4,4,4,4,4,4,4,4,4


In [11]:
#remove non eeg channels
Epilepsy2 = [i.iloc[:,1:14] for i in  Epilepsy] ### maybe 15
Control2 = [i.iloc[:,1:14] for i in  Control]
Epilepsy[0].head()

,Unnamed: 0,AF3,AF4,F3,F4,F7,F8,FC5,FC6,O1,...,CQ_F3,CQ_P7,CQ_P8,CQ_F4,CQ_AF3,CQ_FC5,CQ_O1,CQ_T8,CQ_F8,CQ_DRL
0,1,4036,330.221053,4134,-213.473684,155.242105,3712,322.421053,3935,4169,...,4,4,4,4,4,4,4,4,4,4
1,2,4053,341.505263,4138,-211.936842,160.368421,3715,321.905263,3944,4173,...,4,4,4,4,4,4,4,4,4,4
2,3,4053,-344.757895,4137,-212.957895,156.778947,3713,322.421053,3939,4174,...,4,4,4,4,4,4,4,4,4,4
3,4,4044,343.557895,4136,-212.452632,147.042105,3713,323.452632,3930,4170,...,4,4,4,4,4,4,4,4,4,4
4,5,4049,-344.242105,4131,-213.989474,141.400000,3712,319.347368,3927,4170,...,4,4,4,4,4,4,4,4,4,4


In [12]:
%%capture
#Convert each dataframe to mne object
Epilepsy3 =[convertDF2MNE(i) for i in  Epilepsy2]
Control3 = [convertDF2MNE(i) for i in  Control2]

In [13]:
%%capture
#concatenate the epochs
Epilepsy_epochs = mne.concatenate_epochs(Epilepsy3)
Control_epochs = mne.concatenate_epochs(Control3)

In [14]:
Epilepsy_group=np.concatenate([[i]*len(Epilepsy3[i]) for i in range(len(Epilepsy3))])#create a list of list where each sub list corresponds to subject_no
Control_group=np.concatenate([[i]*len(Control3[i]) for i in range(len(Control3))])#create a list of list where each sub list corresponds to subject_no

Epilepsy_label=np.concatenate([[0]*len(Epilepsy3[i]) for i in range(len(Epilepsy3))])
Control_label=np.concatenate([[1]*len(Control3[i]) for i in range(len(Control3))])

In [15]:
#combine data
data=mne.concatenate_epochs([Epilepsy_epochs,Control_epochs])
group=np.concatenate((Epilepsy_group,Control_group))
label=np.concatenate((Epilepsy_label,Control_label))
print(len(data),len(group),len(label))

Not setting metadata
6127 matching events found
No baseline correction applied
0 bad epochs dropped
6127 6127 6127


/usr/local/lib/python3.12/dist-packages/mne/epochs.py:435: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  selected = np.where(np.in1d(self.events[:, 2], values))[0]
/usr/local/lib/python3.12/dist-packages/mne/epochs.py:460: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  sub = np.where(np.in1d(selection, self.selection))[0]


In [16]:
print(mne.__version__)

1.0.0


In [ ]:
# source: https://mne.tools/stable/auto_tutorials/clinical/60_sleep.html#sphx-glr-auto-tutorials-clinical-60-sleep-py
from mne.time_frequency import psd_welch
def eeg_power_band(epochs):
    """EEG relative power band feature extraction.

    This function takes an ``mne.Epochs`` object and creates EEG features basedg
    on relative power in specific frequency bands that are compatible with
    scikit-learn.

    Parameters
    ----------
    epochs : Epochs
        The data.

    Returns
    -------
    X : numpy array of shape [n_samples, 5]
        Transformed data.
    """
    # specific frequency bands
    FREQ_BANDS = {"delta": [0.5, 4],
                  "theta": [4, 8],
                  "alpha": [8, 16],
                  # "sigma": [11.5, 15.5],
                  "beta": [16, 32],
                  # "gamma": [30, 45],
                  }

    psds, freqs = psd_welch(epochs, picks='eeg', fmin=1, fmax=30)# Compute the PSD using the Welch method
    psds /= np.sum(psds, axis=-1, keepdims=True)    # Normalize the PSDs

    X = []#For each frequency band, compute the mean PSD in that band
    for fmin, fmax in FREQ_BANDS.values():
        psds_band = psds[:, :, (freqs >= fmin) & (freqs < fmax)].mean(axis=-1)# Compute the mean PSD in each frequency band.
        X.append(psds_band)

    return np.concatenate(X, axis=1)#Concatenate the mean PSDs for each band into a single feature vector

In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [19]:
%%capture
features=[]
for d in range(len(data)):#get features from each epoch and save in a list
  features.append(eeg_power_band(data[d]))

In [20]:
#convert list to array
features=np.concatenate(features)
features.shape

(6127, 52)

In [21]:
label.shape

(6127,)

In [22]:
#do 5 fold cross validation
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import classification_report,confusion_matrix, accuracy_score, mean_squared_error
classifier = RandomForestClassifier(n_estimators=40, criterion="entropy",random_state = 42)    # criterion: default="gini"
acc_tracker = []

X_train, X_test, y_train, y_test = train_test_split(features,label,test_size=0.2,random_state = 42)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.fit_transform(X_test)
classifier.fit(X_train,y_train)
y_train_pred = classifier.predict(X_train)
y_pred = classifier.predict(X_test)
train_accuracy = accuracy_score(y_train,y_train_pred)
accuracy = accuracy_score(y_test,y_pred)
mse_train = mean_squared_error(y_train, y_train_pred)
mse = mean_squared_error(y_test, y_pred)
print(mse,accuracy)

0.3083197389885807 0.6916802610114192


# logistic r

In [23]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(max_iter=1000, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.2, random_state=42)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

classifier.fit(X_train, y_train)

y_train_pred = classifier.predict(X_train)
y_pred = classifier.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
accuracy = accuracy_score(y_test, y_pred)

mse_train = mean_squared_error(y_train, y_train_pred)
mse = mean_squared_error(y_test, y_pred)

print(mse, accuracy)

0.28629690048939643 0.7137030995106036


# Support Vector Machine (SVM)

In [24]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error

classifier = SVC(kernel='rbf', random_state=42)

X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.2, random_state=42)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

classifier.fit(X_train, y_train)

y_train_pred = classifier.predict(X_train)
y_pred = classifier.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
accuracy = accuracy_score(y_test, y_pred)

mse_train = mean_squared_error(y_train, y_train_pred)
mse = mean_squared_error(y_test, y_pred)

print(mse, accuracy)

0.2406199021207178 0.7593800978792822


## Grid search svm

In [25]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error

# Split data
X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.2, random_state=42)

# Scale data
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Model
svc = SVC()

# Parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.01, 0.001],
    'kernel': ['rbf', 'linear', 'poly', 'sigmoid'],
    'degree': [2, 3, 4]  # used when kernel='poly'
}

# Grid Search
grid = GridSearchCV(
    estimator=svc,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Train with Grid Search
grid.fit(X_train, y_train)

# Best model
best_model = grid.best_estimator_

# Predictions
y_train_pred = best_model.predict(X_train)
y_pred = best_model.predict(X_test)

# Metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_pred)

mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_pred)

print("Best Parameters:", grid.best_params_)
print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)
print("Train MSE:", mse_train)
print("Test MSE:", mse_test)

Fitting 5 folds for each of 192 candidates, totalling 960 fits
Best Parameters: {'C': 10, 'degree': 2, 'gamma': 0.01, 'kernel': 'rbf'}
Train Accuracy: 0.8437053662517854
Test Accuracy: 0.7601957585644372
Train MSE: 0.15629463374821465
Test MSE: 0.2398042414355628


# KNN

In [26]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error

classifier = KNeighborsClassifier(n_neighbors=5)

X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.2, random_state=42)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

classifier.fit(X_train, y_train)

y_train_pred = classifier.predict(X_train)
y_pred = classifier.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
accuracy = accuracy_score(y_test, y_pred)

mse_train = mean_squared_error(y_train, y_train_pred)
mse = mean_squared_error(y_test, y_pred)

print(mse, accuracy)

0.35970636215334423 0.6402936378466558


## Grid search knn

In [27]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.2, random_state=42)

# Feature scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Base model
knn = KNeighborsClassifier()

# Hyperparameter grid
param_grid = {
    'n_neighbors': [3,5,7,9,11,15],
    'weights': ['uniform','distance'],
    'metric': ['euclidean','manhattan','minkowski'],
    'p': [1,2]   # used for minkowski
}

# Grid Search
grid = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Train grid search
grid.fit(X_train, y_train)

# Best model
best_knn = grid.best_estimator_

# Predictions
y_train_pred = best_knn.predict(X_train)
y_pred = best_knn.predict(X_test)

# Metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_pred)

mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_pred)

print("Best Parameters:", grid.best_params_)
print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)
print("Train MSE:", mse_train)
print("Test MSE:", mse_test)

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Best Parameters: {'metric': 'manhattan', 'n_neighbors': 9, 'p': 1, 'weights': 'distance'}
Train Accuracy: 1.0
Test Accuracy: 0.6508972267536705
Train MSE: 0.0
Test MSE: 0.34910277324632955


# Decision Tree

In [28]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error

classifier = DecisionTreeClassifier(criterion="entropy", random_state=42)

X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.2, random_state=42)

sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

classifier.fit(X_train, y_train)

y_train_pred = classifier.predict(X_train)
y_pred = classifier.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
accuracy = accuracy_score(y_test, y_pred)

mse_train = mean_squared_error(y_train, y_train_pred)
mse = mean_squared_error(y_test, y_pred)

print(mse, accuracy)

0.3841761827079935 0.6158238172920065


## Grid search for DT


In [29]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, mean_squared_error

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.2, random_state=42)

# Scaling (not required for trees but keeping your structure)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Base model
dt = DecisionTreeClassifier(random_state=42)

# Hyperparameter grid
param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'splitter': ['best', 'random'],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': [None, 'sqrt', 'log2']
}

# Grid Search
grid = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Train
grid.fit(X_train, y_train)

# Best model
best_dt = grid.best_estimator_

# Predictions
y_train_pred = best_dt.predict(X_train)
y_pred = best_dt.predict(X_test)

# Metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_pred)

mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_pred)

print("Best Parameters:", grid.best_params_)
print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)
print("Train MSE:", mse_train)
print("Test MSE:", mse_test)

Fitting 5 folds for each of 810 candidates, totalling 4050 fits
Best Parameters: {'criterion': 'gini', 'max_depth': 10, 'max_features': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'splitter': 'random'}
Train Accuracy: 0.7598449296062029
Test Accuracy: 0.664763458401305
Train MSE: 0.2401550703937972
Test MSE: 0.3352365415986949


## comparing models + best model



In [30]:
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score

# # Models
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import SVC
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.naive_bayes import GaussianNB

# # Train-test split
# X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.2, random_state=42)

# # Feature scaling
# sc = StandardScaler()
# X_train = sc.fit_transform(X_train)
# X_test = sc.transform(X_test)

# # Models dictionary
# models = {
#     "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
#     "Logistic Regression": LogisticRegression(max_iter=1000),
#     "SVM": SVC(kernel='rbf'),
#     "KNN": KNeighborsClassifier(n_neighbors=5),
#     "Decision Tree": DecisionTreeClassifier(criterion="entropy"),

# }

# # Accuracy comparison
# results = {}

# for name, model in models.items():

#     model.fit(X_train, y_train)
#     y_pred = model.predict(X_test)

#     accuracy = accuracy_score(y_test, y_pred)

#     results[name] = accuracy

#     print(name, "Accuracy:", accuracy)

# # Print best model
# best_model = max(results, key=results.get)
# print("\nBest Model:", best_model, "with accuracy:", results[best_model])

Random Forest Accuracy: 0.7275693311582382
Logistic Regression Accuracy: 0.7137030995106036
SVM Accuracy: 0.7593800978792822
KNN Accuracy: 0.6402936378466558
Decision Tree Accuracy: 0.6305057096247961

Best Model: SVM with accuracy: 0.7593800978792822


## hybrid  style (Chat GPT )

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Train test split
X_train, X_test, y_train, y_test = train_test_split(features, label, test_size=0.2, random_state=42)

# Pipeline
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC())   # placeholder model
])

# Parameter grid for multiple models
param_grid = [

    {
        'model': [SVC()],
        'model__C': [0.1,1,10,100],
        'model__gamma': ['scale','auto',0.01,0.001],
        'model__kernel': ['rbf','linear']
    },

    {
        'model': [KNeighborsClassifier()],
        'model__n_neighbors': [3,5,7,9,11],
        'model__weights': ['uniform','distance'],
        'model__metric': ['euclidean','manhattan']
    },

    {
        'model': [DecisionTreeClassifier()],
        'model__criterion': ['gini','entropy'],
        'model__max_depth': [None,5,10,15],
        'model__min_samples_split': [2,5,10]
    }

]

# Grid search
grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Train
grid.fit(X_train, y_train)

# Best model
best_model = grid.best_estimator_

# Prediction
y_pred = best_model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Best Model:", grid.best_params_['model'])
print("Best Parameters:", grid.best_params_)
print("Best Accuracy:", accuracy)

Fitting 5 folds for each of 76 candidates, totalling 380 fits
Best Model: SVC()
Best Parameters: {'model': SVC(), 'model__C': 10, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
Best Accuracy: 0.7601957585644372
